# 0. Title and Overview

# Biomedical Image Registration and Deformation Fields

## From intensity alignment to lightweight VoxelMorph-style deformation reasoning

Biomedical image registration asks how to spatially align images that were acquired at different times, with different devices, under different motion states, or from different biological samples. In medical imaging and microscopy, misalignment can make a healthy tissue look diseased, make a therapy response look larger or smaller than it is, or cause a robot or automated biological laboratory system to act on the wrong spatial location.

This notebook is designed for **Google Colab** and also runs on a local CPU environment. It is CPU-compatible by default and automatically uses a GPU when one is available. The notebook follows **Concept first, code second**: each major section explains the idea before showing the implementation.

Students will implement similarity metrics, a brute-force translation baseline, a tiny learned deformation-field model inspired by VoxelMorph, lightweight training on synthetic biomedical-like images, quantitative evaluation, visualization, ablation, and failure analysis. The synthetic data are small and deterministic, while a small public image from `skimage.data` is used when available to keep the lesson grounded in real scientific image examples.

### What you will do

| Activity | Purpose |
|---|---|
| Load a small public scientific image when available | Connect the task to real biomedical/scientific image data |
| Generate tiny synthetic image pairs | Create known deformation fields for controlled learning and evaluation |
| Implement MSE and normalized cross-correlation | Understand common intensity-based similarity metrics |
| Implement a translation-search baseline | See why simple methods are useful but limited |
| Train a tiny deformation-field CNN | Learn the core idea behind learning-based registration without large training |
| Visualize deformation fields and residual errors | Interpret registration output rather than treating it as a black box |
| Run ablation and failure tests | Think like a researcher about robustness and limits |

### Table of contents

0. Title and Overview  
1. Colab Setup and Runtime Check  
2. Learning Goals  
3. Background and Motivation  
4. Mathematical Core  
5. Synthetic Dataset Generator  
6. Baseline Method  
7. Lightweight Learned or Research-Inspired Method  
8. Lightweight Training Loop  
9. Evaluation Metrics  
10. Visualization and Interpretation  
11. Ablation or Stress Test  
12. Failure Modes and Limitations  
13. Optional Full-Scale Training Resources  
14. Research Extension Mini-Project  
15. Exercises and Reflection Questions  
16. Summary and Further Reading  

### Key references and URLs

- scikit-image example data: https://scikit-image.org/docs/stable/api/skimage.data.html
- SimpleITK registration example: https://simpleitk.readthedocs.io/en/master/link_ImageRegistrationMethod1_docs.html
- VoxelMorph repository: https://github.com/voxelmorph/voxelmorph
- VoxelMorph paper: https://arxiv.org/abs/1802.02604
- SimpleITK project: https://simpleitk.org/

No large datasets, login-required resources, paid APIs, or pretrained checkpoints are downloaded by default.

# 1. Colab Setup and Runtime Check

This section sets a deterministic seed, chooses the runtime device, and imports only lightweight libraries. The required default is `FAST_MODE = True`, which keeps the dataset, model, and number of training epochs small enough for Colab CPU. If Colab provides a GPU, the same code automatically uses it.

The setup cell is intentionally explicit because reproducibility is part of scientific computing: students should know when a model uses CPU or GPU, what seed was used, and which package versions are present.

In [ ]:
import random
import math
import platform
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
FAST_MODE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# These flags favor deterministic behavior when possible.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Platform:", platform.platform())
print("FAST_MODE:", FAST_MODE)

# 2. Learning Goals

By the end of this notebook, you should be able to:

1. Explain why biomedical image registration is needed for longitudinal, multimodal, and motion-sensitive analysis.
2. Distinguish rigid, affine, and deformable registration and identify when each transformation class is appropriate.
3. Implement mean squared error and normalized cross-correlation as lightweight image similarity metrics.
4. Generate small synthetic fixed and moving image pairs with known deformation fields for controlled experiments.
5. Implement a brute-force translation-search baseline and explain why it is reasonable but limited.
6. Train a tiny deformation-field CNN for a few epochs on synthetic data and connect it to VoxelMorph-style learning-based registration.
7. Evaluate registration using MSE, NCC, Dice, IoU, PSNR, landmark-style displacement error, and deformation smoothness.
8. Diagnose failure cases caused by contrast changes, missing structures, unrealistic deformation, or weak similarity cues.

# 3. Background and Motivation

Biomedical images are rarely perfectly aligned. A tumor scan acquired today may be shifted relative to last month's scan because the patient was positioned differently. A microscopy time-lapse may drift because the stage moved slightly. Tissue sections can stretch, tear, fold, or shrink during preparation. In automated biological laboratory systems, a visual model may need to align images of wells, tissue, cells, or instruments before measuring morphology or choosing an action.

Naive pixel-wise comparison fails when images are shifted, rotated, stretched, or locally deformed. A pixel at location $(x, y)$ in one image may correspond to a nearby but different location in another image. Registration tries to estimate this correspondence.

Registration is impactful for longitudinal disease monitoring, radiotherapy planning, image-guided intervention, atlas construction, microscopy drift correction, tissue-section alignment, and robotic or AutoBio systems that need spatial grounding. However, registration methods make assumptions: corresponding anatomy or structures exist, the similarity metric is meaningful, interpolation is acceptable, and the estimated deformation is physically plausible.

In practice, pay attention to similarity metrics, transformation models, interpolation, deformation smoothness, missing structures, intensity changes, and failure modes.

In [ ]:
def normalize01(image, eps=1e-8):
    """Normalize an image to the range [0, 1]."""
    image = np.asarray(image, dtype=np.float32)
    image = image - np.nanmin(image)
    image = image / (np.nanmax(image) + eps)
    return image.astype(np.float32)


def center_crop_or_resize(image, size=128):
    """Center-crop an image to a square. If it is too small, pad with zeros."""
    image = np.asarray(image)
    if image.ndim == 3:
        image = image[..., 0]
    h, w = image.shape[:2]
    crop = min(h, w, size)
    y0 = max((h - crop) // 2, 0)
    x0 = max((w - crop) // 2, 0)
    cropped = image[y0:y0 + crop, x0:x0 + crop]
    if cropped.shape[0] < size or cropped.shape[1] < size:
        padded = np.zeros((size, size), dtype=cropped.dtype)
        padded[:cropped.shape[0], :cropped.shape[1]] = cropped
        cropped = padded
    return cropped[:size, :size]


def fallback_synthetic_image(size=128):
    """Create a simple synthetic biomedical-like image if public example data are unavailable."""
    rng = np.random.default_rng(SEED)
    y, x = np.mgrid[0:size, 0:size]
    image = np.zeros((size, size), dtype=np.float32)
    for _ in range(7):
        cy = rng.uniform(0.2 * size, 0.8 * size)
        cx = rng.uniform(0.2 * size, 0.8 * size)
        sigma = rng.uniform(4, 12)
        amplitude = rng.uniform(0.5, 1.2)
        image += amplitude * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * sigma ** 2))
    return normalize01(image)


def load_public_scientific_image(size=128):
    """Load a small public scientific image from scikit-image when available."""
    try:
        from skimage import data
        if hasattr(data, "cell"):
            image = data.cell()
            source = "skimage.data.cell()"
        elif hasattr(data, "camera"):
            image = data.camera()
            source = "skimage.data.camera() fallback"
        else:
            image = fallback_synthetic_image(size=size)
            source = "synthetic fallback"
    except Exception as exc:
        print("scikit-image example data unavailable; using synthetic fallback.")
        print("Reason:", repr(exc))
        image = fallback_synthetic_image(size=size)
        source = "synthetic fallback"
    image = center_crop_or_resize(image, size=size)
    image = normalize01(image)
    return image, source

fixed_public, public_source = load_public_scientific_image(size=128)
# A simple circular shift is used only for the motivational display in this section.
moving_public = np.roll(fixed_public, shift=(8, -6), axis=(0, 1))

print("Public or fallback image source:", public_source)
print("Fixed image shape:", fixed_public.shape, "intensity range:", (float(fixed_public.min()), float(fixed_public.max())))

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, image, title in zip(
    axes,
    [fixed_public, moving_public, np.abs(fixed_public - moving_public)],
    ["Fixed public example", "Moving shifted example", "Absolute difference"]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 4. Mathematical Core

The central idea of image registration is to find a spatial transform that maps locations in the fixed image to corresponding locations in the moving image.

- $I_f$ is the fixed image, the reference image.
- $I_m$ is the moving image, the image that will be warped.
- $T$ is a spatial transform that tells us where to sample from $I_m$.
- $I_m \circ T$ is the warped moving image after applying the transform.
- $\mathcal{D}$ is an image mismatch or dissimilarity measure.
- $\mathcal{R}$ is a regularization penalty that discourages unrealistic transformations.
- $\lambda$ controls the trade-off between image matching and smoothness.

The registration objective is often written as:

$$
T^* = \arg\min_T \mathcal{D}(I_f, I_m \circ T) + \lambda \mathcal{R}(T)
$$

This means that the best transform $T^*$ should make the warped moving image look like the fixed image, while avoiding deformation fields that are too noisy, folded, or physically implausible.

For a dense deformable registration field, we often write the transform as:

$$
T(x, y) = (x + u_x(x, y), y + u_y(x, y))
$$

Here, $u_x$ and $u_y$ are displacement components at every pixel. A smooth field means neighboring pixels move in similar directions. A non-smooth field may overfit image noise or create anatomically impossible warps.

The implementation below defines MSE, normalized cross-correlation, and a differentiable PyTorch warping function. The same warping function will be used by the baseline, the learned model, and evaluation.

In [ ]:
def mse_np(a, b):
    """Mean squared error for NumPy arrays."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.mean((a - b) ** 2))


def ncc_np(a, b, eps=1e-8):
    """Normalized cross-correlation for NumPy arrays."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    a_centered = a - a.mean()
    b_centered = b - b.mean()
    numerator = np.mean(a_centered * b_centered)
    denominator = np.sqrt(np.mean(a_centered ** 2) * np.mean(b_centered ** 2)) + eps
    return float(numerator / denominator)


def warp_image_torch(image, displacement_pixels, mode="bilinear", padding_mode="border"):
    """Warp an image using a dense displacement field in pixel units.

    Parameters
    ----------
    image:
        Tensor with shape (B, 1, H, W).
    displacement_pixels:
        Tensor with shape (B, 2, H, W). Channel 0 is vertical displacement dy,
        and channel 1 is horizontal displacement dx. For each output pixel, the
        function samples the input image at (y + dy, x + dx).
    """
    if image.ndim != 4 or displacement_pixels.ndim != 4:
        raise ValueError("Expected image shape (B, 1, H, W) and displacement shape (B, 2, H, W).")
    batch_size, _, height, width = image.shape
    dtype = image.dtype
    device_local = image.device

    y_coords, x_coords = torch.meshgrid(
        torch.arange(height, dtype=dtype, device=device_local),
        torch.arange(width, dtype=dtype, device=device_local),
        indexing="ij"
    )
    y_coords = y_coords.unsqueeze(0).expand(batch_size, -1, -1)
    x_coords = x_coords.unsqueeze(0).expand(batch_size, -1, -1)

    sample_y = y_coords + displacement_pixels[:, 0]
    sample_x = x_coords + displacement_pixels[:, 1]

    grid_x = 2.0 * sample_x / max(width - 1, 1) - 1.0
    grid_y = 2.0 * sample_y / max(height - 1, 1) - 1.0
    grid = torch.stack([grid_x, grid_y], dim=-1)

    return F.grid_sample(image, grid, mode=mode, padding_mode=padding_mode, align_corners=True)


def numpy_to_image_tensor(image, device_target=device):
    """Convert a 2D NumPy image to a PyTorch tensor with shape (1, 1, H, W)."""
    return torch.from_numpy(np.asarray(image, dtype=np.float32))[None, None].to(device_target)


def displacement_to_tensor(displacement, device_target=device):
    """Convert a displacement array with shape (2, H, W) to tensor shape (1, 2, H, W)."""
    return torch.from_numpy(np.asarray(displacement, dtype=np.float32))[None].to(device_target)

# Sanity check: create a known moving image and then undo the displacement.
with torch.no_grad():
    fixed_tensor = numpy_to_image_tensor(fixed_public, device)
    known_disp = np.zeros((2, fixed_public.shape[0], fixed_public.shape[1]), dtype=np.float32)
    known_disp[0] = 5.0   # dy
    known_disp[1] = -4.0  # dx
    known_disp_tensor = displacement_to_tensor(known_disp, device)
    moving_tensor = warp_image_torch(fixed_tensor, -known_disp_tensor)
    corrected_tensor = warp_image_torch(moving_tensor, known_disp_tensor)

moving_known = moving_tensor.cpu().numpy()[0, 0]
corrected_known = corrected_tensor.cpu().numpy()[0, 0]

print("Before correction MSE:", mse_np(fixed_public, moving_known))
print("After correction MSE:", mse_np(fixed_public, corrected_known))
print("Before correction NCC:", ncc_np(fixed_public, moving_known))
print("After correction NCC:", ncc_np(fixed_public, corrected_known))

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, image, title in zip(
    axes,
    [fixed_public, moving_known, corrected_known],
    ["Fixed image", "Moving image", "After known correction"]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 5. Synthetic Dataset Generator

Synthetic data are used for lightweight training because real clinical registration datasets can be large, restricted, or difficult to validate. Synthetic data also give us known deformation fields, which lets us measure whether an estimated field is close to the ground truth.

This is not a replacement for real validation. It is a controlled classroom environment. The generated images are small, deterministic, and biomedical-like: they contain bright blob-like structures, soft boundaries, and occasional ring-like patterns. The moving image is created by warping the fixed image with a known smooth displacement field, then applying small intensity and noise perturbations.

The defaults below are intentionally small:

```python
if FAST_MODE:
    num_train = 64
    num_val = 16
    image_size = 64
    num_epochs = 5
else:
    num_train = 256
    num_val = 64
    image_size = 96
    num_epochs = 10
```

In [ ]:
num_train = 64 if FAST_MODE else 256
num_val = 16 if FAST_MODE else 64
image_size = 64 if FAST_MODE else 96
num_epochs = 5 if FAST_MODE else 10

batch_size = 8 if FAST_MODE else 16
learning_rate = 1e-3

print("Dataset settings:", {"num_train": num_train, "num_val": num_val, "image_size": image_size, "num_epochs": num_epochs})

def make_synthetic_biomedical_image(size, rng, max_blobs=8):
    """Create a small biomedical-like 2D image from Gaussian blobs and rings."""
    y, x = np.mgrid[0:size, 0:size]
    image = np.zeros((size, size), dtype=np.float32)
    num_blobs = int(rng.integers(4, max_blobs + 1))
    for _ in range(num_blobs):
        cy = rng.uniform(0.15 * size, 0.85 * size)
        cx = rng.uniform(0.15 * size, 0.85 * size)
        sigma_y = rng.uniform(2.0, 7.0)
        sigma_x = rng.uniform(2.0, 7.0)
        amplitude = rng.uniform(0.4, 1.2)
        image += amplitude * np.exp(-(((y - cy) ** 2) / (2 * sigma_y ** 2) + ((x - cx) ** 2) / (2 * sigma_x ** 2)))

    # Add one soft ring-like structure to mimic membranes or boundaries.
    cy = rng.uniform(0.25 * size, 0.75 * size)
    cx = rng.uniform(0.25 * size, 0.75 * size)
    radius = rng.uniform(8, 16) * size / 64
    thickness = rng.uniform(2, 4) * size / 64
    distance = np.sqrt((y - cy) ** 2 + (x - cx) ** 2)
    ring = np.exp(-((distance - radius) ** 2) / (2 * thickness ** 2))
    image += rng.uniform(0.2, 0.7) * ring
    return normalize01(image)


def make_smooth_displacement_field(size, rng, max_translation=5.0, deform_strength=2.5):
    """Create a smooth displacement field in pixel units with shape (2, H, W)."""
    y, x = np.mgrid[0:size, 0:size].astype(np.float32)
    y_norm = y / max(size - 1, 1)
    x_norm = x / max(size - 1, 1)

    translation_y = rng.uniform(-max_translation, max_translation)
    translation_x = rng.uniform(-max_translation, max_translation)
    phase_a = rng.uniform(0, 2 * np.pi)
    phase_b = rng.uniform(0, 2 * np.pi)
    frequency = rng.uniform(1.0, 2.0)

    dy = translation_y + deform_strength * np.sin(2 * np.pi * frequency * x_norm + phase_a) * np.sin(np.pi * y_norm)
    dx = translation_x + deform_strength * np.cos(2 * np.pi * frequency * y_norm + phase_b) * np.sin(np.pi * x_norm)
    return np.stack([dy, dx], axis=0).astype(np.float32)


def apply_intensity_noise(image, rng, noise_std=0.025, contrast_range=(0.85, 1.15), bias_range=(-0.05, 0.05)):
    """Apply small deterministic intensity perturbations."""
    contrast = rng.uniform(*contrast_range)
    bias = rng.uniform(*bias_range)
    noisy = contrast * image + bias + rng.normal(0.0, noise_std, size=image.shape).astype(np.float32)
    return np.clip(noisy, 0.0, 1.0).astype(np.float32)


def generate_registration_pair(size, rng, deform_strength=2.5, noise_std=0.025, missing_structure=False, contrast_inversion=False):
    """Generate one fixed image, moving image, ground-truth displacement field, and mask."""
    fixed = make_synthetic_biomedical_image(size, rng)
    true_disp = make_smooth_displacement_field(size, rng, max_translation=4.5, deform_strength=deform_strength)

    with torch.no_grad():
        fixed_t = numpy_to_image_tensor(fixed, torch.device("cpu"))
        disp_t = displacement_to_tensor(true_disp, torch.device("cpu"))
        # The moving image is made by warping fixed with the inverse field.
        moving = warp_image_torch(fixed_t, -disp_t).cpu().numpy()[0, 0]

    moving = apply_intensity_noise(moving, rng, noise_std=noise_std)

    if contrast_inversion:
        moving = 1.0 - moving

    if missing_structure:
        y0 = int(0.35 * size)
        y1 = int(0.65 * size)
        x0 = int(0.35 * size)
        x1 = int(0.65 * size)
        moving[y0:y1, x0:x1] *= 0.1

    fixed_mask = (fixed > 0.35).astype(np.float32)
    return fixed.astype(np.float32), moving.astype(np.float32), true_disp.astype(np.float32), fixed_mask


def build_dataset(num_samples, size, seed, deform_strength=2.5, noise_std=0.025):
    """Build a deterministic dataset of synthetic registration pairs."""
    rng = np.random.default_rng(seed)
    fixed_list, moving_list, disp_list, mask_list = [], [], [], []
    for _ in range(num_samples):
        fixed, moving, disp, mask = generate_registration_pair(
            size=size,
            rng=rng,
            deform_strength=deform_strength,
            noise_std=noise_std
        )
        fixed_list.append(fixed)
        moving_list.append(moving)
        disp_list.append(disp)
        mask_list.append(mask)
    fixed_array = np.stack(fixed_list)[:, None]
    moving_array = np.stack(moving_list)[:, None]
    disp_array = np.stack(disp_list)
    mask_array = np.stack(mask_list)[:, None]
    return fixed_array, moving_array, disp_array, mask_array

train_fixed, train_moving, train_disp, train_masks = build_dataset(num_train, image_size, SEED + 1)
val_fixed, val_moving, val_disp, val_masks = build_dataset(num_val, image_size, SEED + 2)

assert train_fixed.shape == (num_train, 1, image_size, image_size)
assert train_moving.shape == train_fixed.shape
assert train_disp.shape == (num_train, 2, image_size, image_size)
assert val_fixed.shape == (num_val, 1, image_size, image_size)

train_dataset = TensorDataset(
    torch.from_numpy(train_fixed),
    torch.from_numpy(train_moving),
    torch.from_numpy(train_disp),
    torch.from_numpy(train_masks),
)
val_dataset = TensorDataset(
    torch.from_numpy(val_fixed),
    torch.from_numpy(val_moving),
    torch.from_numpy(val_disp),
    torch.from_numpy(val_masks),
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Train fixed shape:", train_fixed.shape)
print("Validation fixed shape:", val_fixed.shape)
print("Ground-truth displacement range:", float(train_disp.min()), float(train_disp.max()))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row in range(2):
    idx = row
    images = [train_fixed[idx, 0], train_moving[idx, 0], train_masks[idx, 0], np.sqrt(train_disp[idx, 0] ** 2 + train_disp[idx, 1] ** 2)]
    titles = ["Fixed synthetic image", "Moving synthetic image", "Fixed threshold mask", "True displacement magnitude"]
    for col, (image, title) in enumerate(zip(images, titles)):
        axes[row, col].imshow(image, cmap="gray")
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

# 6. Baseline Method

A simple and surprisingly useful baseline is brute-force translation search. It tries many small integer shifts and selects the shift that gives the best similarity score. This is reasonable when the main motion is a global stage drift, scanner offset, or patient positioning change.

However, translation search assumes that every pixel moves by the same amount. It cannot represent rotation, scaling, local tissue deformation, organ motion, microscopy stretching, or missing structures. It also assumes that the intensity relationship is meaningful for the chosen metric.

The code below searches over a small grid of candidate translations, warps the moving image for each candidate, and selects the best candidate by normalized cross-correlation.

In [ ]:
def constant_displacement(height, width, dy, dx, device_target=device):
    """Create a constant displacement tensor with shape (1, 2, H, W)."""
    disp = torch.zeros((1, 2, height, width), dtype=torch.float32, device=device_target)
    disp[:, 0] = float(dy)
    disp[:, 1] = float(dx)
    return disp


def warp_translation_integer_np(image, dy, dx):
    """Fast integer translation warp with border padding.

    This matches the registration convention used in warp_image_torch: each output
    pixel samples the input image at (y + dy, x + dx). Because the candidate
    shifts are integers, NumPy indexing is much faster than repeatedly calling
    a differentiable sampler inside the brute-force baseline.
    """
    image = np.asarray(image, dtype=np.float32)
    height, width = image.shape
    y_indices = np.clip(np.arange(height) + int(dy), 0, height - 1)
    x_indices = np.clip(np.arange(width) + int(dx), 0, width - 1)
    return image[np.ix_(y_indices, x_indices)].astype(np.float32)


def register_by_translation_search(fixed_np, moving_np, search_radius=6, metric="ncc"):
    """Register moving to fixed using brute-force integer translation search."""
    fixed_np = np.asarray(fixed_np, dtype=np.float32)
    moving_np = np.asarray(moving_np, dtype=np.float32)

    best_score = -np.inf if metric == "ncc" else np.inf
    best_shift = (0, 0)
    best_image = moving_np.copy()

    for dy in range(-search_radius, search_radius + 1):
        for dx in range(-search_radius, search_radius + 1):
            warped = warp_translation_integer_np(moving_np, dy, dx)
            score = ncc_np(fixed_np, warped) if metric == "ncc" else mse_np(fixed_np, warped)
            is_better = score > best_score if metric == "ncc" else score < best_score
            if is_better:
                best_score = score
                best_shift = (dy, dx)
                best_image = warped
    return best_image.astype(np.float32), best_shift, float(best_score)

example_index = 0
baseline_registered, baseline_shift, baseline_score = register_by_translation_search(
    val_fixed[example_index, 0],
    val_moving[example_index, 0],
    search_radius=6,
    metric="ncc"
)

print("Best baseline shift (dy, dx):", baseline_shift)
print("Best baseline NCC score:", baseline_score)
print("Unregistered MSE:", mse_np(val_fixed[example_index, 0], val_moving[example_index, 0]))
print("Baseline registered MSE:", mse_np(val_fixed[example_index, 0], baseline_registered))

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, title in zip(
    axes,
    [
        val_fixed[example_index, 0],
        val_moving[example_index, 0],
        baseline_registered,
        np.abs(val_fixed[example_index, 0] - baseline_registered),
    ],
    ["Fixed image", "Moving image", "Translation baseline", "Baseline difference map"]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 7. Lightweight Learned or Research-Inspired Method

Modern deformable registration methods estimate a displacement field rather than a single global transform. VoxelMorph-style systems learn a neural network that predicts a dense field from a fixed-moving image pair. The field is then used to warp the moving image toward the fixed image.

This notebook implements a tiny linear low-resolution deformation-field predictor. It is not a clinical registration system and does not reproduce full VoxelMorph. The simplifications are intentional:

- The images are small 2D synthetic examples.
- Training uses a few dozen examples and a few full-batch optimization steps.
- The model views a heavily downsampled fixed-moving pair and predicts a coarse displacement grid.
- The coarse grid is upsampled to form a dense deformation field.
- Active training uses the known synthetic displacement field as lightweight supervision. Full VoxelMorph-style registration often uses image similarity and smoothness without ground-truth deformation fields.

Why can this improve over the baseline? A deformation-field predictor has many spatial degrees of freedom, so it can represent local motion that a single global translation cannot. In this deliberately tiny notebook, the learned model may still underperform the translation baseline on some examples; that outcome is useful because it shows that model capacity, training data, and validation matter.

The core idea is still faithful: predict a displacement field, warp the moving image, and evaluate whether the warped image aligns with the fixed image. The model is intentionally small so students can focus on deformation-field reasoning rather than heavy architecture engineering.

Full-scale research would require large validated datasets, 3D volumes, stronger regularization, uncertainty estimation, anatomical constraints, careful hyperparameter tuning, and clinical or biological validation.

In [ ]:
class TinyLinearFieldRegressor:
    """A tiny linear predictor for a coarse 2D displacement grid.

    This class uses PyTorch tensors on the selected device, but it avoids heavy
    autograd training. The model is a one-layer linear map from downsampled
    fixed-moving image features to low-resolution displacement vectors.
    """
    def __init__(self, grid_size=8, max_disp=7.0, device_target=device):
        self.grid_size = int(grid_size)
        self.max_disp = float(max_disp)
        self.device = device_target
        self.feature_dim = 3 * self.grid_size * self.grid_size + 1
        self.output_dim = 2 * self.grid_size * self.grid_size
        self.weights = torch.zeros((self.feature_dim, self.output_dim), dtype=torch.float32, device=self.device)

    def eval(self):
        """Compatibility no-op so the object behaves like a tiny model."""
        return self

    def featurize(self, fixed, moving):
        """Create low-resolution fixed, moving, and difference features."""
        fixed_low = F.interpolate(fixed, size=(self.grid_size, self.grid_size), mode="bilinear", align_corners=False)
        moving_low = F.interpolate(moving, size=(self.grid_size, self.grid_size), mode="bilinear", align_corners=False)
        diff_low = fixed_low - moving_low
        features = torch.cat([fixed_low, moving_low, diff_low], dim=1).flatten(start_dim=1)
        bias = torch.ones((features.shape[0], 1), dtype=features.dtype, device=features.device)
        return torch.cat([features, bias], dim=1)

    def low_res_target(self, displacement):
        """Downsample a dense displacement field to the model grid."""
        low = F.interpolate(displacement, size=(self.grid_size, self.grid_size), mode="bilinear", align_corners=False)
        return low.flatten(start_dim=1)

    def predict_low_flat(self, fixed, moving):
        """Predict flattened low-resolution displacement values."""
        features = self.featurize(fixed, moving)
        return torch.clamp(features @ self.weights, -self.max_disp, self.max_disp)

    def __call__(self, fixed, moving):
        """Predict a dense displacement field at the input image resolution."""
        low_flat = self.predict_low_flat(fixed, moving)
        low = low_flat.view(-1, 2, self.grid_size, self.grid_size)
        dense = F.interpolate(low, size=fixed.shape[-2:], mode="bilinear", align_corners=False)
        return dense

    def fit_step(self, features, targets, step_size=0.5, weight_decay=1e-4):
        """Run one full-batch gradient descent step for linear regression."""
        predictions = features @ self.weights
        error = predictions - targets
        mse_loss = (error ** 2).mean()
        regularization = weight_decay * (self.weights[:-1] ** 2).mean()
        # The gradient is for mean squared error over all samples and outputs.
        grad = (2.0 / error.numel()) * (features.T @ error)
        grad[:-1] += (2.0 * weight_decay / max(self.weights[:-1].numel(), 1)) * self.weights[:-1]
        self.weights -= step_size * grad
        return float((mse_loss + regularization).detach().cpu())

    def loss(self, features, targets, weight_decay=1e-4):
        """Evaluate the linear regression objective."""
        predictions = features @ self.weights
        mse_loss = (predictions - targets).pow(2).mean()
        regularization = weight_decay * (self.weights[:-1] ** 2).mean()
        return float((mse_loss + regularization).detach().cpu())


def smoothness_loss(displacement):
    """Penalize large spatial changes in the displacement field."""
    dy = displacement[:, :, 1:, :] - displacement[:, :, :-1, :]
    dx = displacement[:, :, :, 1:] - displacement[:, :, :, :-1]
    return (dy.pow(2).mean() + dx.pow(2).mean())


def endpoint_error(predicted_disp, true_disp):
    """Mean endpoint error between predicted and true displacement fields."""
    return torch.sqrt(((predicted_disp - true_disp) ** 2).sum(dim=1) + 1e-8).mean()

model = TinyLinearFieldRegressor(grid_size=8, max_disp=7.0, device_target=device)
num_parameters = model.weights.numel()
print("Model type: TinyLinearFieldRegressor")
print("Feature dimension:", model.feature_dim)
print("Output dimension:", model.output_dim)
print("Number of trainable parameters:", num_parameters)
print("Model tensor device:", model.weights.device)

# 8. Lightweight Training Loop

For active training, we use the known synthetic deformation field. This keeps the notebook fast and stable on CPU while still teaching how a learned model can output a deformation field.

The lightweight supervised coarse-field loss is:

$$
\mathcal{L} = \operatorname{MSE}(\bar{u}_\theta, \bar{u}_{true}) + \alpha \lVert W \rVert_2^2
$$

Here, $\bar{u}_\theta$ is the predicted coarse displacement grid, $\bar{u}_{true}$ is the synthetic ground-truth field downsampled to the same grid, and $W$ contains the linear model weights. After training, the coarse field is upsampled to the image resolution, used to warp the moving image, and evaluated with image-similarity metrics.

This differs from full unsupervised VoxelMorph-style training, where the main loss is usually image similarity plus smoothness. That full version is more research-realistic but can be slower and more sensitive to implementation details, so it is left as a study resource rather than a required Colab computation.

In [ ]:
# Prepare full-batch feature matrices on the selected device.
train_fixed_t = torch.from_numpy(train_fixed).to(device)
train_moving_t = torch.from_numpy(train_moving).to(device)
train_disp_t = torch.from_numpy(train_disp).to(device)
val_fixed_t = torch.from_numpy(val_fixed).to(device)
val_moving_t = torch.from_numpy(val_moving).to(device)
val_disp_t = torch.from_numpy(val_disp).to(device)

with torch.no_grad():
    train_features = model.featurize(train_fixed_t, train_moving_t)
    train_targets = model.low_res_target(train_disp_t)
    val_features = model.featurize(val_fixed_t, val_moving_t)
    val_targets = model.low_res_target(val_disp_t)

field_learning_rate = 0.8
weight_decay = 1e-4
train_losses = []
val_losses = []

start_time = time.time()
for epoch in range(1, num_epochs + 1):
    train_loss = model.fit_step(train_features, train_targets, step_size=field_learning_rate, weight_decay=weight_decay)
    val_loss = model.loss(val_features, val_targets, weight_decay=weight_decay)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch:02d}/{num_epochs} | train coarse-field loss {train_loss:.4f} | val coarse-field loss {val_loss:.4f}")

elapsed = time.time() - start_time
print(f"Training completed in {elapsed:.3f} seconds on {device}.")

plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o", label="Training coarse-field loss")
plt.plot(range(1, num_epochs + 1), val_losses, marker="s", label="Validation coarse-field loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Lightweight deformation-field training curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Note for GPU users:**
If you have access to a GPU, you can set `FAST_MODE = False`, increase `num_epochs`, increase the dataset size, or replace the tiny model with a stronger architecture. The current notebook keeps training small so that every student can run it in Colab.

# 9. Evaluation Metrics

Registration should be evaluated quantitatively and visually. No single metric is sufficient.

- **MSE** measures pixel-wise intensity mismatch. Lower is better, but it can be sensitive to intensity scaling.
- **NCC** measures correlation after removing mean intensity. Higher is better and can be more robust to simple brightness or contrast scaling.
- **Dice** and **IoU** compare thresholded structures. They are useful when a simple structure mask is meaningful.
- **PSNR** is a signal-quality metric derived from MSE. Higher is better.
- **Endpoint error** measures how close the predicted displacement field is to the known synthetic displacement field.
- **Smoothness** measures how rapidly the field changes. Lower is smoother, but too much smoothness can underfit real motion.

The comparison below evaluates three conditions: unregistered moving image, translation baseline, and the learned deformation-field model.

In [ ]:
def dice_score_np(pred_mask, true_mask, eps=1e-8):
    """Dice score for binary masks."""
    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)
    intersection = np.logical_and(pred_mask, true_mask).sum()
    return float((2 * intersection + eps) / (pred_mask.sum() + true_mask.sum() + eps))


def iou_score_np(pred_mask, true_mask, eps=1e-8):
    """Intersection over union for binary masks."""
    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    return float((intersection + eps) / (union + eps))


def psnr_np(a, b, eps=1e-8):
    """Peak signal-to-noise ratio for images in [0, 1]."""
    error = mse_np(a, b)
    return float(20 * np.log10(1.0 / np.sqrt(error + eps)))


def displacement_smoothness_np(displacement):
    """Mean squared finite-difference smoothness for a displacement array."""
    dy = displacement[:, 1:, :] - displacement[:, :-1, :]
    dx = displacement[:, :, 1:] - displacement[:, :, :-1]
    return float(np.mean(dy ** 2) + np.mean(dx ** 2))


def evaluate_registered_image(fixed_np, registered_np, true_mask_np):
    """Compute image and mask metrics for one registered image."""
    pred_mask = registered_np > 0.35
    return {
        "MSE": mse_np(fixed_np, registered_np),
        "NCC": ncc_np(fixed_np, registered_np),
        "Dice": dice_score_np(pred_mask, true_mask_np > 0.5),
        "IoU": iou_score_np(pred_mask, true_mask_np > 0.5),
        "PSNR": psnr_np(fixed_np, registered_np),
    }


def learned_register_numpy(fixed_np, moving_np):
    """Use the trained model to register one NumPy image pair."""
    model.eval()
    with torch.no_grad():
        fixed_t = numpy_to_image_tensor(fixed_np, device)
        moving_t = numpy_to_image_tensor(moving_np, device)
        pred_disp_t = model(fixed_t, moving_t)
        registered_t = warp_image_torch(moving_t, pred_disp_t)
    registered = registered_t.cpu().numpy()[0, 0].astype(np.float32)
    pred_disp = pred_disp_t.cpu().numpy()[0].astype(np.float32)
    return registered, pred_disp


eval_count = min(8, num_val)
method_records = {"Unregistered": [], "Translation baseline": [], "Tiny learned deformation": []}
epe_records = {"Translation baseline": [], "Tiny learned deformation": []}
smoothness_records = {"Translation baseline": [], "Tiny learned deformation": []}

for idx in range(eval_count):
    fixed_np = val_fixed[idx, 0]
    moving_np = val_moving[idx, 0]
    true_disp_np = val_disp[idx]
    true_mask_np = val_masks[idx, 0]

    unregistered_metrics = evaluate_registered_image(fixed_np, moving_np, true_mask_np)
    method_records["Unregistered"].append(unregistered_metrics)

    baseline_img, best_shift, _ = register_by_translation_search(fixed_np, moving_np, search_radius=6, metric="ncc")
    baseline_disp = np.zeros_like(true_disp_np)
    baseline_disp[0] = best_shift[0]
    baseline_disp[1] = best_shift[1]
    method_records["Translation baseline"].append(evaluate_registered_image(fixed_np, baseline_img, true_mask_np))
    epe_records["Translation baseline"].append(float(np.mean(np.sqrt(np.sum((baseline_disp - true_disp_np) ** 2, axis=0)))))
    smoothness_records["Translation baseline"].append(displacement_smoothness_np(baseline_disp))

    learned_img, learned_disp = learned_register_numpy(fixed_np, moving_np)
    method_records["Tiny learned deformation"].append(evaluate_registered_image(fixed_np, learned_img, true_mask_np))
    epe_records["Tiny learned deformation"].append(float(np.mean(np.sqrt(np.sum((learned_disp - true_disp_np) ** 2, axis=0)))))
    smoothness_records["Tiny learned deformation"].append(displacement_smoothness_np(learned_disp))

metric_names = ["MSE", "NCC", "Dice", "IoU", "PSNR"]
summary_rows = []
for method_name, records in method_records.items():
    row = {"Method": method_name}
    for metric_name in metric_names:
        row[metric_name] = float(np.mean([record[metric_name] for record in records]))
    if method_name in epe_records:
        row["Endpoint error"] = float(np.mean(epe_records[method_name]))
        row["Smoothness"] = float(np.mean(smoothness_records[method_name]))
    else:
        row["Endpoint error"] = float("nan")
        row["Smoothness"] = float("nan")
    summary_rows.append(row)

print("Metric summary over", eval_count, "validation examples:")
header = ["Method", "MSE", "NCC", "Dice", "IoU", "PSNR", "Endpoint error", "Smoothness"]
print(" | ".join(header))
for row in summary_rows:
    print(
        f"{row['Method']} | {row['MSE']:.4f} | {row['NCC']:.4f} | {row['Dice']:.4f} | "
        f"{row['IoU']:.4f} | {row['PSNR']:.2f} | {row['Endpoint error']:.3f} | {row['Smoothness']:.4f}"
    )

plt.figure(figsize=(7, 4))
methods = [row["Method"] for row in summary_rows]
values = [row["MSE"] for row in summary_rows]
plt.bar(methods, values)
plt.ylabel("Mean squared error")
plt.title("Validation MSE comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
values = [row["NCC"] for row in summary_rows]
plt.bar(methods, values)
plt.ylabel("Normalized cross-correlation")
plt.title("Validation NCC comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# 10. Visualization and Interpretation

Visualization helps reveal whether numerical improvements are meaningful. A lower MSE can still correspond to an anatomically implausible warp, especially when the image contrast changes or structures are missing. For registration, inspect the fixed image, moving image, registered outputs, difference maps, deformation arrows, and residual error maps.

The residual error map below is a simple proxy for uncertainty: regions with high residual error are places where the model did not explain the fixed-moving mismatch well. It is not calibrated uncertainty, but it is a useful warning signal for this toy notebook.

In [ ]:
def plot_displacement_arrows(ax, displacement, step=8, title="Displacement field"):
    """Plot a sparse arrow field from a dense displacement array with shape (2, H, W)."""
    height, width = displacement.shape[1:]
    y, x = np.mgrid[0:height:step, 0:width:step]
    dy = displacement[0, ::step, ::step]
    dx = displacement[1, ::step, ::step]
    magnitude = np.sqrt(dx ** 2 + dy ** 2)
    ax.imshow(magnitude, cmap="gray")
    ax.quiver(x, y, dx, dy, angles="xy", scale_units="xy", scale=1.0)
    ax.set_title(title)
    ax.axis("off")

viz_idx = 1 if num_val > 1 else 0
fixed_viz = val_fixed[viz_idx, 0]
moving_viz = val_moving[viz_idx, 0]
true_disp_viz = val_disp[viz_idx]
baseline_viz, baseline_shift_viz, _ = register_by_translation_search(fixed_viz, moving_viz, search_radius=6, metric="ncc")
learned_viz, learned_disp_viz = learned_register_numpy(fixed_viz, moving_viz)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
images = [
    fixed_viz,
    moving_viz,
    baseline_viz,
    learned_viz,
    np.abs(fixed_viz - moving_viz),
    np.abs(fixed_viz - baseline_viz),
    np.abs(fixed_viz - learned_viz),
    np.sqrt(learned_disp_viz[0] ** 2 + learned_disp_viz[1] ** 2),
]
titles = [
    "Fixed image",
    "Moving image",
    "Translation baseline",
    "Tiny learned deformation",
    "Error before registration",
    "Error after baseline",
    "Error after learned method",
    "Learned displacement magnitude",
]
for ax, image, title in zip(axes.ravel(), images, titles):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_displacement_arrows(axes[0], true_disp_viz, step=8, title="True displacement field")
baseline_disp_viz = np.zeros_like(true_disp_viz)
baseline_disp_viz[0] = baseline_shift_viz[0]
baseline_disp_viz[1] = baseline_shift_viz[1]
plot_displacement_arrows(axes[1], baseline_disp_viz, step=8, title="Baseline constant field")
plot_displacement_arrows(axes[2], learned_disp_viz, step=8, title="Learned displacement field")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
residual_proxy = np.abs(fixed_viz - learned_viz)
plt.imshow(residual_proxy, cmap="gray")
plt.title("Residual error proxy map")
plt.axis("off")
plt.colorbar(label="Absolute residual")
plt.show()

## Interpretation

Better alignment should reduce structured error in the difference maps. The translation baseline can correct global drift but cannot follow spatially varying deformation. The learned deformation field can represent local motion, but it may become unstable when similarity cues are weak, when the synthetic training distribution is too narrow, or when the model is too small.

A visually plausible warp is not automatically biologically or clinically valid. In real biomedical use, registration output should be checked against anatomical landmarks, downstream measurements, expert review, and known physical constraints.

# 11. Ablation or Stress Test

Ablation asks how performance changes when one controlled factor changes. Here we vary the deformation strength in newly generated validation pairs while keeping the trained model fixed. The translation baseline should work best when the motion is mostly global. As local deformation becomes stronger, the baseline should struggle because it has only two parameters: one vertical shift and one horizontal shift.

The learned method has a richer transform model, but it can still fail if the deformation is outside the training distribution or if the images lack enough local structure.

In [ ]:
stress_strengths = [0.0, 1.5, 3.0, 4.5, 6.0] if FAST_MODE else [0.0, 1.5, 3.0, 4.5, 6.0, 7.5]
stress_repeats = 3 if FAST_MODE else 5
stress_rng = np.random.default_rng(SEED + 100)

stress_results = {
    "deform_strength": [],
    "unregistered_mse": [],
    "baseline_mse": [],
    "learned_mse": [],
    "unregistered_ncc": [],
    "baseline_ncc": [],
    "learned_ncc": [],
}

for strength in stress_strengths:
    unreg_mse_values, base_mse_values, learned_mse_values = [], [], []
    unreg_ncc_values, base_ncc_values, learned_ncc_values = [], [], []
    for _ in range(stress_repeats):
        fixed_s, moving_s, _, _ = generate_registration_pair(
            size=image_size,
            rng=stress_rng,
            deform_strength=strength,
            noise_std=0.025
        )
        base_s, _, _ = register_by_translation_search(fixed_s, moving_s, search_radius=6, metric="ncc")
        learned_s, _ = learned_register_numpy(fixed_s, moving_s)

        unreg_mse_values.append(mse_np(fixed_s, moving_s))
        base_mse_values.append(mse_np(fixed_s, base_s))
        learned_mse_values.append(mse_np(fixed_s, learned_s))
        unreg_ncc_values.append(ncc_np(fixed_s, moving_s))
        base_ncc_values.append(ncc_np(fixed_s, base_s))
        learned_ncc_values.append(ncc_np(fixed_s, learned_s))

    stress_results["deform_strength"].append(strength)
    stress_results["unregistered_mse"].append(float(np.mean(unreg_mse_values)))
    stress_results["baseline_mse"].append(float(np.mean(base_mse_values)))
    stress_results["learned_mse"].append(float(np.mean(learned_mse_values)))
    stress_results["unregistered_ncc"].append(float(np.mean(unreg_ncc_values)))
    stress_results["baseline_ncc"].append(float(np.mean(base_ncc_values)))
    stress_results["learned_ncc"].append(float(np.mean(learned_ncc_values)))

plt.figure(figsize=(7, 4))
plt.plot(stress_results["deform_strength"], stress_results["unregistered_mse"], marker="o", label="Unregistered")
plt.plot(stress_results["deform_strength"], stress_results["baseline_mse"], marker="s", label="Translation baseline")
plt.plot(stress_results["deform_strength"], stress_results["learned_mse"], marker="^", label="Tiny learned deformation")
plt.xlabel("Synthetic deformation strength")
plt.ylabel("Mean squared error")
plt.title("Stress test: MSE versus deformation strength")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(stress_results["deform_strength"], stress_results["unregistered_ncc"], marker="o", label="Unregistered")
plt.plot(stress_results["deform_strength"], stress_results["baseline_ncc"], marker="s", label="Translation baseline")
plt.plot(stress_results["deform_strength"], stress_results["learned_ncc"], marker="^", label="Tiny learned deformation")
plt.xlabel("Synthetic deformation strength")
plt.ylabel("Normalized cross-correlation")
plt.title("Stress test: NCC versus deformation strength")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Stress-test results:")
for i, strength in enumerate(stress_results["deform_strength"]):
    print(
        f"Strength {strength:.1f}: baseline MSE {stress_results['baseline_mse'][i]:.4f}, "
        f"learned MSE {stress_results['learned_mse'][i]:.4f}, "
        f"baseline NCC {stress_results['baseline_ncc'][i]:.4f}, "
        f"learned NCC {stress_results['learned_ncc'][i]:.4f}"
    )

## Stress-test interpretation

When deformation strength is low, a global translation may explain much of the mismatch. As local deformation grows, a translation-only model becomes less appropriate. The learned method has the capacity to model local displacement, but it is limited by small data, few epochs, synthetic assumptions, and a tiny network. If performance degrades at high deformation levels, the most likely reason is distribution shift: the test deformation is more difficult than what the model learned.

# 12. Failure Modes and Limitations

Failure analysis matters because biomedical AI systems can appear to work while violating biological or clinical assumptions. A registration algorithm may reduce image difference by moving structures into visually convenient locations, even if those correspondences are anatomically wrong.

This section creates a synthetic failure case with two violations:

1. **Contrast inversion:** bright structures become dark and dark structures become bright, breaking an intensity-similarity assumption.
2. **Missing structure:** part of the moving image is removed, breaking the assumption that corresponding structures exist in both images.

The model was not trained for this domain shift, and the baseline metric is also not designed for it.

In [ ]:
failure_rng = np.random.default_rng(SEED + 500)
fixed_fail, moving_fail, disp_fail, mask_fail = generate_registration_pair(
    size=image_size,
    rng=failure_rng,
    deform_strength=3.0,
    noise_std=0.03,
    missing_structure=True,
    contrast_inversion=True,
)

baseline_fail, baseline_fail_shift, _ = register_by_translation_search(fixed_fail, moving_fail, search_radius=6, metric="ncc")
learned_fail, learned_fail_disp = learned_register_numpy(fixed_fail, moving_fail)

print("Failure case metrics:")
print("Unregistered MSE:", mse_np(fixed_fail, moving_fail), "NCC:", ncc_np(fixed_fail, moving_fail))
print("Baseline MSE:", mse_np(fixed_fail, baseline_fail), "NCC:", ncc_np(fixed_fail, baseline_fail))
print("Learned MSE:", mse_np(fixed_fail, learned_fail), "NCC:", ncc_np(fixed_fail, learned_fail))
print("Baseline failure shift (dy, dx):", baseline_fail_shift)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
images = [
    fixed_fail,
    moving_fail,
    baseline_fail,
    learned_fail,
    np.abs(fixed_fail - moving_fail),
    np.abs(fixed_fail - baseline_fail),
    np.abs(fixed_fail - learned_fail),
    np.sqrt(learned_fail_disp[0] ** 2 + learned_fail_disp[1] ** 2),
]
titles = [
    "Fixed failure image",
    "Moving with inversion and missing structure",
    "Baseline output",
    "Learned output",
    "Failure error before registration",
    "Failure error after baseline",
    "Failure error after learned method",
    "Failure predicted displacement magnitude",
]
for ax, image, title in zip(axes.ravel(), images, titles):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Failure-mode interpretation

The method failed because the intensity relationship and correspondence assumptions were violated. With contrast inversion, MSE and NCC no longer reward the same anatomical match as in the training data. With missing structure, there may be no correct local correspondence for part of the image.

A lower image-difference score can still be anatomically wrong. A researcher might try mutual information for multimodal intensity changes, feature-based registration, stronger deformation regularization, anatomical landmarks, segmentation-aware losses, uncertainty estimation, inverse-consistency constraints, or training data that includes realistic domain shifts.

# 13. Optional Full-Scale Training Resources

This notebook does not run full-scale training. Full-scale registration research often requires 3D volumes, validated datasets, larger models, careful preprocessing, stronger regularization, and extensive evaluation. Use the resources below for further study, but do not download large files automatically inside this notebook.

- Original VoxelMorph paper: https://arxiv.org/abs/1802.02604  
  Introduces an unsupervised learning framework for fast deformable medical image registration.
- Official VoxelMorph code repository: https://github.com/voxelmorph/voxelmorph  
  Contains research code and examples for learning-based registration.
- SimpleITK registration example: https://simpleitk.readthedocs.io/en/master/link_ImageRegistrationMethod1_docs.html  
  Demonstrates classical intensity-based image registration workflows.
- SimpleITK project: https://simpleitk.org/  
  Provides tools for medical image analysis and registration.
- scikit-image public example data: https://scikit-image.org/docs/stable/api/skimage.data.html  
  Provides small scientific images useful for teaching and lightweight experiments.
- Learn2Reg challenge information: https://learn2reg.grand-challenge.org/  
  A challenge ecosystem for medical image registration research.
- OASIS brain imaging data: https://www.oasis-brains.org/  
  A public neuroimaging resource that can be relevant for full-scale registration studies.

Optional full-scale code sketch, commented out by default:

```python
# This optional full-scale training sketch is commented out by default.
# It is not required for this notebook and may involve larger downloads or longer runtimes.
# GPU users who want a real VoxelMorph workflow should follow the official repository.
#
# !git clone https://github.com/voxelmorph/voxelmorph.git
# %cd voxelmorph
# # Read the repository documentation before running training scripts.
# # Prepare a validated dataset, choose an experiment configuration, and monitor runtime.
```

# 14. Research Extension Mini-Project

## Hypothesis

A similarity metric based on normalized cross-correlation is more robust than mean squared error under simple intensity scaling, but both metrics become unreliable under contrast inversion or missing structures.

## What to modify

- Change the baseline similarity metric from NCC to MSE.
- Vary contrast scaling, noise level, blur, missing structure size, and deformation strength.
- Change the translation search range.
- Change the smoothness weight in the learned model.
- Try setting `FAST_MODE = False` if you have GPU access.

## What to measure

- MSE
- NCC
- Dice and IoU of thresholded structures
- Endpoint error when synthetic ground truth is available
- Deformation smoothness
- Failure-case frequency across random seeds

## Expected result

NCC should outperform MSE under simple brightness or contrast scaling because it removes the mean and normalizes intensity variation. Both methods may fail under severe structural changes or contrast inversion.

## Possible negative result

A metric improvement may not correspond to anatomically correct alignment. The algorithm may exploit interpolation or smoothness assumptions to reduce image mismatch while producing biologically implausible correspondences.

## How to report findings

Include a table of metric values, visual examples of successes and failures, displacement-field plots, and a short discussion of whether the method would be safe for biomedical or AutoBio deployment. GPU is recommended only if you increase dataset size, training epochs, or model capacity.

# 15. Exercises and Reflection Questions

## Part A: Conceptual Understanding

1. What is the difference between a fixed image and a moving image?
2. When is rigid registration sufficient, and when is affine or deformable registration needed?
3. What does each vector in a dense deformation field represent?
4. Why does regularization matter in deformable registration?
5. Why can registration be clinically risky even when the image difference looks small?
6. What assumptions are made when using MSE as a similarity metric?
7. What assumptions are made when using NCC as a similarity metric?

## Part B: Implementation Understanding

1. How does `warp_image_torch` convert pixel displacement into normalized grid coordinates?
2. Why does interpolation matter during image warping?
3. How does the translation baseline search over candidate shifts?
4. Why does the tiny CNN output two channels rather than one channel?
5. What role does the smoothness loss play during training?
6. How do the number of epochs, dataset size, and deformation strength affect results?
7. Why is the synthetic endpoint-error term not always available in real biomedical registration?

## Part C: Research Thinking

1. How would you validate registration without a ground-truth deformation field?
2. What might go wrong when registering multimodal images such as MRI and CT?
3. Why might microscopy images violate smooth deformation assumptions?
4. How could an AutoBio system be harmed by a poor visual registration step?
5. What uncertainty estimates would help a human reviewer decide whether to trust a warp?
6. How would this notebook need to change for 3D medical volumes?
7. What extra safeguards are needed before using registration output in clinical decision-making?
8. How does this tiny model relate to VoxelMorph-style learning-based registration, and what is missing?

# 16. Summary and Further Reading

In this notebook, you learned why biomedical image registration matters, how intensity-based similarity metrics work, why a translation baseline is useful but limited, and how a tiny learned model can predict a dense deformation field. You implemented synthetic data generation, image warping, MSE, NCC, Dice, IoU, PSNR, endpoint error, smoothness scoring, a translation-search baseline, lightweight deformation-field training, visualization, stress testing, and failure analysis.

Registration matters because many biomedical measurements require spatial correspondence across time points, modalities, tissue sections, microscope fields, or robotic observations. Deformation fields matter because they are not just technical outputs: they encode hypotheses about how tissue, cells, organs, or instruments moved.

Use rigid or translation registration when motion is global and simple. Use deformable registration when local motion, tissue deformation, anatomical variability, or microscopy drift requires spatially varying displacement. Do not trust registration output when structures are missing, contrast relationships change, the deformation is physically implausible, the validation set is weak, or the downstream decision is high stakes.

What to study next:

- VoxelMorph paper: https://arxiv.org/abs/1802.02604
- VoxelMorph code: https://github.com/voxelmorph/voxelmorph
- SimpleITK registration documentation: https://simpleitk.readthedocs.io/en/master/link_ImageRegistrationMethod1_docs.html
- SimpleITK project: https://simpleitk.org/
- scikit-image example data: https://scikit-image.org/docs/stable/api/skimage.data.html
- Learn2Reg challenge: https://learn2reg.grand-challenge.org/
- OASIS brain imaging data: https://www.oasis-brains.org/

For research practice, always pair registration metrics with visual inspection, anatomical knowledge, uncertainty analysis, and downstream validation.